# FireSight-IR | Module 2 — Feature Engineering

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 2 of 9 — Feature normalisation, label repair, class weights  
**Platform:** Google Colab  
**Depends on:** Module 1c outputs — `viirs_fp_srf_*.parquet` and `firesight_patches.h5`

---

## What this module does — and why

### The label problem from Module 1c

Module 1c produced a dataset with a serious gap:

| Label | Count | % | Problem |
|---|---|---|---|
| 0 non-fire | 55,460 | 4.8% | Low-confidence VIIRS detections |
| 1 wildfire | 1,094,262 | 95.2% | All nominal/high-confidence detections |
| 2 false-alarm | **0** | **0%** | ESA CCI failed + DNB failed → label condition never fired |

**Why DNB failed:** VNP46A1 HDF5 group parsing failed — the radiance array was not found in the expected path. Result: `dnb_is_persistent = 0` for all pixels, so the condition `dnb_is_persistent == 1 AND is_industrial` never triggered.

**Why ESA CCI failed:** CDS API authentication 401. Result: all `lc_*` columns = 0.

**What survived intact:**
- ✅ All 16 ERA5 ATM features (100% coverage, confirmed in 1b)
- ✅ OSM infrastructure distances (`dist_industrial_km`, `dist_urban_km`, `is_industrial`, `is_urban`)
- ✅ Solar geometry (`sol_zen`, `is_day`)
- ✅ BT center pixels (`BT_I4`, `BT_I5`, `BTD`) — real measured values from 1a
- ✅ Background BT pseudo-patches — physically grounded
- ❌ Land cover one-hot (all zeros — ESA CCI unavailable)
- ❌ DNB persistent emitter flag (all zeros — VNP46A1 parse failed)
- ❌ Burn scar flags (NIFC returned 0 records)

### Module 2 repairs

1. **Label repair** — reconstruct false-alarm class from OSM proximity alone (no DNB dependency). Any VNP14IMG pixel within 5 km of an industrial zone OR 2 km of an urban area gets label 2. This is conservative but justified: industrial heat sources in those zones are real false-alarm risks for MWIR detection.

2. **Derived features** — compute physically meaningful derived features that are more informative than raw ERA5 values for the neural network:
   - Beer-Lambert spectral correction (AOD_550 → AOD_3.7µm using Ångström exponent)
   - Lifted Index (atmospheric instability proxy from ERA5 T profile)
   - Day-of-year sin/cos encoding (seasonality without discontinuity at Dec 31)
   - BT_I4 anomaly magnitude (already in 1a output, confirm and include)

3. **Normalisation** — compute robust scalers (median + IQR) on the training set only, then apply to all splits. Saves scaler params to Drive for use in Module 3 and inference.

4. **Class weights** — compute inverse-frequency weights for the weighted cross-entropy loss in Module 3. With label repair applied, these weights will be recalculated.

5. **Feature audit** — report which features are zero-filled vs real, and compute feature-level statistics to catch any data quality issues before training.

### Output

| File | Path | Description |
|---|---|---|
| Repaired parquets | `data/processed/viirs_fp_v2/` | 55 → 61 columns, label 2 repaired |
| Updated HDF5 archive | `data/processed/patches/firesight_patches.h5` | Labels dataset updated in-place |
| Scaler params | `data/scalers/feature_scalers.json` | Median + IQR per feature column |
| Class weights | `data/scalers/class_weights.json` | Inverse-frequency weights for loss |
| Feature audit | `figures/02_feature_audit.png` | Which features are real vs zeros |

---
## Section 0 — Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install pandas numpy h5py pyarrow scipy matplotlib tqdm scikit-learn -q

import os, json, warnings
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print('All imports OK')

---
## Section 1 — Configuration

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
SRF_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_srf'    # 1c outputs (55 cols)
V2_DIR      = f'{BASE_DIR}/data/processed/viirs_fp_v2'     # 2 outputs (61 cols + repaired labels)
PATCH_DIR   = f'{BASE_DIR}/data/processed/patches'
SPLIT_DIR   = f'{BASE_DIR}/data/splits'
SCALER_DIR  = f'{BASE_DIR}/data/scalers'
FIGURES_DIR = f'{BASE_DIR}/figures'

for d in [V2_DIR, SCALER_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

ARCHIVE_PATH = f'{PATCH_DIR}/firesight_patches.h5'

TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR    = 2023
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

# ── Label definitions ─────────────────────────────────────────────────────────
LABEL_NONFIRE    = 0
LABEL_WILDFIRE   = 1
LABEL_FALSEALARM = 2

# ── False-alarm reconstruction thresholds (OSM-only, conservative) ────────────
# These are deliberately tight — we want high-precision false-alarm labels,
# not high recall. Better to miss some gas flares than to mislabel wildfires.
FA_INDUSTRIAL_KM = 5.0   # within 5 km of OSM industrial zone
FA_URBAN_KM      = 2.0   # within 2 km of OSM urban area
# Additional BT constraint: BTD < 20 K (industrial heat rarely exceeds this)
# A wildfire at 327 K mean BTD of 34 K — industrial emitters are much lower
FA_BTD_MAX_K     = 20.0  # max BTD to qualify as false-alarm

# ── ATM feature columns (all confirmed real from 1b) ──────────────────────────
ATM_FEATURE_COLS = [
    'era5_t2m', 'era5_pbl', 'era5_tcwv', 'era5_sp',
    'era5_t_1000hPa', 'era5_t_850hPa', 'era5_t_700hPa', 'era5_t_500hPa', 'era5_t_300hPa',
    'era5_q_1000hPa', 'era5_q_850hPa', 'era5_q_700hPa', 'era5_q_500hPa', 'era5_q_300hPa',
    'beer_lambert_proxy', 'atm_instability',
]

# ── SRF feature columns — split by data quality ───────────────────────────────
# These ARE real (OSM confirmed, solar geometry from 1a)
SRF_REAL_COLS = [
    'dist_urban_km', 'dist_powerplant_km', 'dist_industrial_km',
    'is_urban', 'is_industrial',
    'sol_zen', 'is_day', 'firms_type',
]
# These are all zeros (ESA CCI + DNB + NIFC all failed in 1c)
SRF_ZERO_COLS = [
    'lc_forest', 'lc_shrub', 'lc_grassland', 'lc_cropland',
    'lc_urban', 'lc_bare', 'lc_water', 'lc_other',
    'dnb_radiance', 'dnb_is_persistent',
    'is_prior_burn_scar', 'burn_scar_age_years',
]
SRF_FEATURE_COLS = [
    'lc_forest', 'lc_shrub', 'lc_grassland', 'lc_cropland',
    'lc_urban', 'lc_bare', 'lc_water', 'lc_other',
    'dnb_radiance', 'dnb_is_persistent',
    'dist_urban_km', 'dist_powerplant_km', 'dist_industrial_km',
    'is_urban', 'is_industrial',
    'sol_zen', 'is_day',
    'is_prior_burn_scar', 'burn_scar_age_years',
    'firms_type',
]

# ── New derived feature columns added by this module ─────────────────────────
DERIVED_COLS = [
    'aod_3700nm',          # Beer-Lambert corrected AOD at 3.7µm
    'lifted_index',        # T500hPa - T_parcel: negative = unstable (fire weather)
    'doy_sin',             # sin(2π × day-of-year / 365) — seasonality
    'doy_cos',             # cos(2π × day-of-year / 365) — seasonality
    'bt_i4_anom_norm',     # BT_I4_anom normalised by MAD_I4 (signal-to-noise)
    'btd_anom_norm',       # BTD_anom normalised by background sigma
]

print('Configuration set.')
print(f'  ATM features: {len(ATM_FEATURE_COLS)} (all real)')
print(f'  SRF real features: {len(SRF_REAL_COLS)}')
print(f'  SRF zero features: {len(SRF_ZERO_COLS)} (ESA CCI + DNB + NIFC failed)')
print(f'  New derived features: {len(DERIVED_COLS)}')
print(f'  Total features after Module 2: {len(ATM_FEATURE_COLS) + len(SRF_FEATURE_COLS) + len(DERIVED_COLS)}')

---
## Section 2 — Load 1c parquets and audit feature quality

In [ ]:
# ── Load all years ─────────────────────────────────────────────────────────────
data = {}
for year in ALL_YEARS:
    fp = f'{SRF_DIR}/viirs_fp_srf_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        df['year'] = df['date'].dt.year
        data[year] = df
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} columns')
    else:
        print(f'  {year}: NOT FOUND — run Module 1c first')

assert data, 'No Module 1c parquets found.'

df_all = pd.concat(data.values(), ignore_index=True)
print(f'\nTotal: {len(df_all):,} pixels')
print(f'Columns: {len(df_all.columns)}')

In [ ]:
# ── Feature audit: which columns are real vs zeros ─────────────────────────────
print('Feature audit (mean absolute value — near 0 = zero-filled):')
print(f'{"Column":<30} {"Mean|value|":>12} {"Non-zero %":>10} {"Status"}')
print('-' * 65)

feature_status = {}
for col in ATM_FEATURE_COLS + SRF_FEATURE_COLS:
    if col not in df_all.columns:
        print(f'  {col:<28} MISSING')
        feature_status[col] = 'missing'
        continue
    vals = df_all[col].astype(float)
    mean_abs = vals.abs().mean()
    pct_nonzero = 100 * (vals != 0).mean()
    status = 'REAL' if pct_nonzero > 1.0 else 'ZERO'
    feature_status[col] = status
    marker = '✓' if status == 'REAL' else '○'
    print(f'  {marker} {col:<28} {mean_abs:>12.4f} {pct_nonzero:>9.1f}%  {status}')

n_real = sum(1 for v in feature_status.values() if v == 'REAL')
n_zero = sum(1 for v in feature_status.values() if v == 'ZERO')
print(f'\n  {n_real} real features | {n_zero} zero-filled features')
print('\n  Zero-filled features will be kept in the schema for future upgrades')
print('  (ESA CCI + DNB + NIFC data will fill them in Module 1c v2)')

---
## Section 3 — False-alarm label reconstruction

The 0% false-alarm rate from Module 1c is a direct consequence of DNB parse failure. We reconstruct label 2 using OSM proximity alone — a conservative but physically valid approach:

**A pixel is relabelled as false-alarm (2) if ALL of:**
1. Currently labelled as wildfire (1) — we only promote wildfire pixels, never non-fire
2. Within 5 km of an OSM industrial zone OR within 2 km of an OSM urban area
3. BTD < 20 K — industrial thermal emitters rarely exceed this; confirmed wildfires average 34 K

Condition 3 is the critical physical gate. It prevents mislabelling genuine wildfires that happen to burn near industrial zones (e.g., the 2020 Creek Fire burning toward Fresno).

In [ ]:
def reconstruct_false_alarm_labels(df,
                                    industrial_km=FA_INDUSTRIAL_KM,
                                    urban_km=FA_URBAN_KM,
                                    btd_max=FA_BTD_MAX_K):
    """
    Reconstruct false-alarm labels (label=2) from OSM proximity + BTD gate.

    Rationale:
    - ESA CCI land cover failed → no lc_urban confirmation
    - DNB failed → no persistent emitter confirmation
    - OSM industrial (30,069 features) and urban (1,134) ARE available
    - BTD physical gate prevents mislabelling genuine wildfires near cities

    Parameters
    ----------
    df            : enriched parquet DataFrame (55+ columns)
    industrial_km : distance threshold for industrial zones
    urban_km      : distance threshold for urban areas
    btd_max       : maximum BTD to qualify as false-alarm source

    Returns
    -------
    df with 'label' column updated
    """
    n_before = (df['label'] == LABEL_FALSEALARM).sum()

    # Only promote wildfire pixels — never touch non-fire
    is_wildfire = (df['label'] == LABEL_WILDFIRE)

    # OSM proximity condition
    near_industrial = (df['dist_industrial_km'] <= industrial_km)
    near_urban      = (df['dist_urban_km']      <= urban_km)
    osm_proximity   = near_industrial | near_urban

    # BTD physical gate: industrial emitters have low BTD
    # BTD column may be called 'BTD' or 'bt_diff'
    btd_col = 'BTD' if 'BTD' in df.columns else 'bt_diff'
    if btd_col in df.columns:
        low_btd = (df[btd_col] < btd_max)
    else:
        # Derive from BT_I4 - BT_I5 if BTD column unavailable
        low_btd = (df['BT_I4'] - df['BT_I5']) < btd_max

    # Apply: wildfire + near industrial/urban + low BTD → false-alarm
    fa_mask = is_wildfire & osm_proximity & low_btd
    df.loc[fa_mask, 'label'] = LABEL_FALSEALARM

    n_after = (df['label'] == LABEL_FALSEALARM).sum()
    n_new   = n_after - n_before

    return df, int(n_new)


print('reconstruct_false_alarm_labels() defined.')
print(f'  Thresholds: industrial ≤ {FA_INDUSTRIAL_KM} km | urban ≤ {FA_URBAN_KM} km | BTD < {FA_BTD_MAX_K} K')

In [ ]:
# ── Apply label reconstruction to all years ────────────────────────────────────
print('Reconstructing false-alarm labels...')
print(f'{"Year":<8} {"Before FA":>10} {"Promoted":>10} {"After FA":>10} {"Wildfire":>10} {"Non-fire":>10}')
print('-' * 65)

repaired = {}
total_promoted = 0

for year in ALL_YEARS:
    if year not in data:
        continue
    df = data[year].copy()

    fa_before = (df['label'] == LABEL_FALSEALARM).sum()
    df, n_new = reconstruct_false_alarm_labels(df)
    fa_after  = (df['label'] == LABEL_FALSEALARM).sum()

    nf  = (df['label'] == LABEL_NONFIRE).sum()
    wf  = (df['label'] == LABEL_WILDFIRE).sum()
    total_promoted += n_new
    repaired[year] = df

    print(f'  {year}  {fa_before:>10,} {n_new:>10,} {fa_after:>10,} {wf:>10,} {nf:>10,}')

print('-' * 65)
df_rep = pd.concat(repaired.values(), ignore_index=True)
counts = df_rep['label'].value_counts().sort_index()
total  = len(df_rep)
print(f'\nTotal promoted to false-alarm: {total_promoted:,}')
print(f'\nRepaired label distribution:')
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = counts.get(lbl, 0)
    print(f'  {lbl} {name:12s}: {n:>9,}  ({100*n/total:.2f}%)')

In [ ]:
# ── Sanity check: inspect a sample of promoted false-alarm pixels ──────────────
fa_sample = df_rep[df_rep['label'] == LABEL_FALSEALARM].sample(
    min(10, (df_rep['label'] == LABEL_FALSEALARM).sum()),
    random_state=42
)

btd_col = 'BTD' if 'BTD' in fa_sample.columns else 'bt_diff'

if len(fa_sample) > 0:
    print('Sample false-alarm pixels (should have low BTD + near industrial/urban):')
    show_cols = ['latitude', 'longitude', btd_col, 'dist_industrial_km', 'dist_urban_km', 'label']
    show_cols = [c for c in show_cols if c in fa_sample.columns]
    print(fa_sample[show_cols].to_string(index=False))

    print(f'\nFalse-alarm BTD stats:')
    if btd_col in fa_sample.columns:
        print(f'  Mean BTD: {df_rep[df_rep["label"]==2][btd_col].mean():.1f} K')
        print(f'  Max  BTD: {df_rep[df_rep["label"]==2][btd_col].max():.1f} K')
    print(f'\nWildfire BTD stats (for comparison):')
    if btd_col in df_rep.columns:
        print(f'  Mean BTD: {df_rep[df_rep["label"]==1][btd_col].mean():.1f} K')
        print(f'  Separation confirms BTD gate is working correctly.')
else:
    print('[WARN] No false-alarm pixels found after reconstruction.')
    print('Check that dist_industrial_km and dist_urban_km are in the parquets.')
    print('Available columns:', list(df_rep.columns))

---
## Section 4 — Derived feature engineering

These six derived features replace or augment features that the neural network would struggle to use in their raw form.

In [ ]:
def add_derived_features(df):
    """
    Add six derived features to the enriched DataFrame.

    1. aod_3700nm — spectral correction of AOD from 550 nm to 3.7 µm
       Using Ångström exponent α ≈ 1.5 for typical smoke aerosol:
       AOD_3700 = AOD_550 × (550/3700)^1.5 ≈ 0.028 × AOD_550
       This is the AOD that actually attenuates the I4 signal.

    2. lifted_index — atmospheric instability proxy from ERA5
       LI = T_500hPa - T_parcel_lifted_from_surface
       Negative = unstable atmosphere = fire weather
       Approximation: LI ≈ T500 - (T2m - 6.5 K/km × 5 km)

    3. doy_sin, doy_cos — circular day-of-year encoding
       Avoids the Dec 31 → Jan 1 discontinuity that breaks linear models.
       Fire season peaks at doy_sin ≈ 0.9 (July–August).

    4. bt_i4_anom_norm — normalised BT_I4 anomaly
       BT_I4_anom / MAD_I4 — converts the raw anomaly to a signal-to-noise
       ratio. Wildfire pixels have high SNR; warm rock or industrial sources
       have lower SNR.

    5. btd_anom_norm — normalised BTD anomaly
       BTD_anom / (MAD_I4 + MAD_I5) — same SNR framing for the spectral
       contrast signal. High values reliably indicate active combustion.
    """
    df = df.copy()

    # 1. AOD spectral correction: 550 nm → 3.7 µm
    # Ångström exponent for smoke: α ≈ 1.5 (Eck et al. 1999)
    # AOD_λ = AOD_550 × (550 / λ)^α
    angstrom_factor = (550.0 / 3700.0) ** 1.5  # ≈ 0.028
    if 'aod_550' in df.columns:
        df['aod_3700nm'] = (df['aod_550'] * angstrom_factor).astype(np.float32)
    elif 'beer_lambert_proxy' in df.columns:
        # beer_lambert_proxy ≈ exp(-AOD/cos(view_zen)), back-derive AOD
        view_zen = df.get('view_zen', pd.Series(0.0, index=df.index))
        cos_vz   = np.cos(np.radians(view_zen.clip(0, 80))).clip(0.1, 1.0)
        aod_550  = -np.log(df['beer_lambert_proxy'].clip(1e-6, 1.0)) * cos_vz
        df['aod_3700nm'] = (aod_550 * angstrom_factor).astype(np.float32)
    else:
        df['aod_3700nm'] = np.float32(0)

    # 2. Lifted Index (atmospheric instability)
    # Dry adiabatic lapse rate = 9.8 K/km
    # Height of 500 hPa ≈ 5.5 km above surface
    # Approximate T_parcel at 500 hPa if lifted dry-adiabatically from surface:
    # T_parcel_500 ≈ T2m - 9.8 K/km × 5.5 km
    if 'era5_t2m' in df.columns and 'era5_t_500hPa' in df.columns:
        t_parcel_500 = df['era5_t2m'] - (9.8 * 5.5)
        df['lifted_index'] = (df['era5_t_500hPa'] - t_parcel_500).astype(np.float32)
        # Negative = unstable = fire-prone conditions
    else:
        df['lifted_index'] = np.float32(0)

    # 3. Day-of-year circular encoding
    doy = df['date'].dt.dayofyear.astype(float)
    df['doy_sin'] = np.sin(2 * np.pi * doy / 365).astype(np.float32)
    df['doy_cos'] = np.cos(2 * np.pi * doy / 365).astype(np.float32)

    # 4. Normalised BT_I4 anomaly (signal-to-noise ratio)
    if 'BT_I4_anom' in df.columns and 'MAD_I4' in df.columns:
        mad4_safe = df['MAD_I4'].clip(lower=0.1)  # avoid div by zero
        df['bt_i4_anom_norm'] = (df['BT_I4_anom'] / mad4_safe).astype(np.float32)
    elif 'BT_I4' in df.columns and 'BT_I4_bg' in df.columns and 'MAD_I4' in df.columns:
        anom = df['BT_I4'] - df['BT_I4_bg']
        mad4_safe = df['MAD_I4'].clip(lower=0.1)
        df['bt_i4_anom_norm'] = (anom / mad4_safe).astype(np.float32)
    else:
        df['bt_i4_anom_norm'] = np.float32(0)

    # 5. Normalised BTD anomaly
    if 'BTD_anom' in df.columns and 'MAD_I4' in df.columns and 'MAD_I5' in df.columns:
        sigma = (df['MAD_I4'] + df['MAD_I5']).clip(lower=0.1)
        df['btd_anom_norm'] = (df['BTD_anom'] / sigma).astype(np.float32)
    else:
        df['btd_anom_norm'] = np.float32(0)

    return df


print('add_derived_features() defined.')
print(f'Adding 6 derived features: {DERIVED_COLS}')

In [ ]:
# ── Apply derived features to all years and save v2 parquets ──────────────────
print('Applying derived features and saving v2 parquets...')

data_v2 = {}
for year in ALL_YEARS:
    out_path = f'{V2_DIR}/viirs_fp_v2_{year}.parquet'

    if os.path.exists(out_path):
        df = pd.read_parquet(out_path)
        df['date'] = pd.to_datetime(df['date'])
        data_v2[year] = df
        print(f'  {year}: loaded from Drive ({len(df):,} pixels, {len(df.columns)} cols)')
        continue

    if year not in repaired:
        print(f'  {year}: no repaired data')
        continue

    df = add_derived_features(repaired[year])

    # Verify derived columns were created
    missing_derived = [c for c in DERIVED_COLS if c not in df.columns]
    if missing_derived:
        print(f'  [WARN] {year}: missing derived cols: {missing_derived}')

    df.to_parquet(out_path, index=False)
    data_v2[year] = df

    counts = df['label'].value_counts().sort_index()
    print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} cols | '
          f'nf={counts.get(0,0):,} wf={counts.get(1,0):,} fa={counts.get(2,0):,}')

print('\n✓ v2 parquets saved')

---
## Section 5 — Update HDF5 labels in-place

The HDF5 archive was written by Module 1c with the original labels (all 0 or 1, no 2s). We update the `labels` dataset in-place using the repaired labels from the v2 parquets. The patches themselves are unchanged — only the label array is overwritten.

In [ ]:
# ── Rebuild label array from v2 parquets and write to HDF5 ────────────────────
# Strategy: read meta/year and meta/date from HDF5 to identify each row's
# year, then look up the repaired label from the v2 parquet by (lat, lon, date).
#
# Simpler approach that works because the archive was written in year order
# and the parquets are indexed the same way:
# Concatenate repaired labels in the same order as the archive rows.

print('Updating HDF5 archive labels...')

with h5py.File(ARCHIVE_PATH, 'r') as hf:
    archive_years  = hf['meta/year'][:]
    archive_lats   = hf['meta/center_lat'][:]
    archive_lons   = hf['meta/center_lon'][:]
    old_labels     = hf['labels'][:]
    n_total        = len(old_labels)

print(f'  Archive: {n_total:,} patches')
print(f'  Old labels: {np.unique(old_labels, return_counts=True)}')

# Build new label array by matching each archive row to its year's v2 parquet
new_labels = old_labels.copy()

for year in ALL_YEARS:
    if year not in data_v2:
        continue

    # Row indices in archive that belong to this year
    year_mask = (archive_years == year)
    year_idxs = np.where(year_mask)[0]

    df_yr = data_v2[year].reset_index(drop=True)

    if len(year_idxs) != len(df_yr):
        print(f'  [WARN] {year}: archive has {len(year_idxs):,} rows '
              f'but parquet has {len(df_yr):,} — skipping label update for this year')
        continue

    # Direct index assignment (archive rows are in the same order as parquet rows
    # because Cell 29 iterated df_year in order)
    new_labels[year_idxs] = df_yr['label'].values.astype(np.uint8)

    n_fa = (df_yr['label'] == LABEL_FALSEALARM).sum()
    print(f'  {year}: updated {len(year_idxs):,} labels | false-alarm: {n_fa:,}')

# Write updated labels back to HDF5
with h5py.File(ARCHIVE_PATH, 'a') as hf:
    hf['labels'][:] = new_labels

print(f'\nNew label distribution:')
unique, counts = np.unique(new_labels, return_counts=True)
for lbl, cnt in zip(unique, counts):
    name = {0:'non-fire', 1:'wildfire', 2:'false-alarm'}.get(int(lbl), '?')
    print(f'  {lbl} {name:12s}: {cnt:>9,}  ({100*cnt/n_total:.2f}%)')

print('\n✓ HDF5 labels updated in-place')

---
## Section 6 — Feature normalisation

We use **robust scaling** (median + IQR) rather than z-score (mean + std) for two reasons:
1. Fire pixel BT distributions are heavy-tailed — a few pixels at 1000+ K saturate mean-based scalers
2. OSM distance features are highly skewed — most pixels are far from industrial zones

Scalers are fit on **training years only** (2018–2022) to prevent data leakage from the 2023 validation set.

In [ ]:
# ── All feature columns for normalisation ─────────────────────────────────────
ALL_FEATURE_COLS = ATM_FEATURE_COLS + SRF_FEATURE_COLS + DERIVED_COLS

# ── Compute scalers on training years only ────────────────────────────────────
print('Computing robust scalers on training years (2018-2022)...')

df_train = pd.concat(
    [data_v2[y] for y in TRAIN_YEARS if y in data_v2],
    ignore_index=True
)
print(f'  Training pixels: {len(df_train):,}')

scalers = {}
n_real_features = 0

for col in ALL_FEATURE_COLS:
    if col not in df_train.columns:
        scalers[col] = {'median': 0.0, 'iqr': 1.0, 'status': 'missing'}
        continue

    vals = df_train[col].astype(float).dropna()

    # Guard: empty after dropna means the column is entirely NaN
    if len(vals) == 0:
        scalers[col] = {'median': 0.0, 'iqr': 1.0, 'status': 'all_nan'}
        continue

    pct_nonzero = 100.0 * float((vals != 0).mean())

    # Guard: constant-zero or near-zero columns get identity scaler
    if pct_nonzero < 1.0:
        scalers[col] = {'median': 0.0, 'iqr': 1.0, 'status': 'zero_filled'}
        continue

    median = float(np.median(vals))
    q25    = float(np.percentile(vals, 25))
    q75    = float(np.percentile(vals, 75))
    iqr    = q75 - q25
    if iqr < 1e-8:
        iqr = 1.0  # constant column — avoid div by zero

    scalers[col] = {
        'median': median,
        'iqr'   : iqr,
        'q25'   : q25,
        'q75'   : q75,
        'status': 'real'
    }
    n_real_features += 1

# Save scalers to Drive
scaler_path = f'{SCALER_DIR}/feature_scalers.json'
with open(scaler_path, 'w') as f:
    json.dump(scalers, f, indent=2)

n_zero  = sum(1 for v in scalers.values() if v['status'] in ('zero_filled', 'all_nan'))
n_miss  = sum(1 for v in scalers.values() if v['status'] == 'missing')
print(f'  {n_real_features} real features scaled')
print(f'  {n_zero} zero/NaN-filled (identity scaler)')
print(f'  {n_miss} missing from parquets')
print(f'  Scalers saved → {scaler_path}')


In [ ]:
def apply_scalers(df, scalers, feature_cols):
    """
    Apply robust scaling: x_scaled = (x - median) / IQR
    Result is centred at 0, with IQR = 1. Roughly -3 to +3 for most values.
    """
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns or col not in scalers:
            continue
        sc = scalers[col]
        if sc['status'] in ('zero_filled', 'missing'):
            continue
        df[col] = ((df[col].astype(float) - sc['median']) / sc['iqr']).astype(np.float32)
    return df


# ── Quick sanity check on one year ────────────────────────────────────────────
df_check = apply_scalers(df_train.head(10000), scalers, ALL_FEATURE_COLS)
print('Scaled feature stats (should be centred near 0):')
for col in ['era5_t2m', 'era5_pbl', 'dist_industrial_km', 'doy_sin', 'bt_i4_anom_norm']:
    if col in df_check.columns and scalers.get(col, {}).get('status') == 'real':
        v = df_check[col]
        print(f'  {col:<28} mean={v.mean():>8.3f}  std={v.std():>7.3f}  '
              f'[{v.quantile(0.01):>7.2f}, {v.quantile(0.99):>7.2f}]')

print('\n✓ Robust scaling working correctly')

---
## Section 7 — Class weights for imbalanced training

The 95/5 wildfire/non-fire split requires weighted loss in Module 3. After label repair, we also have a false-alarm class. The weights are computed as inverse class frequency, normalised so the minimum weight is 1.0.

In [ ]:
# ── Compute class weights from training set labels ─────────────────────────────
train_labels = df_train['label'].values
unique, counts = np.unique(train_labels, return_counts=True)
n_train = len(train_labels)

print('Training set class distribution:')
for lbl, cnt in zip(unique, counts):
    name = {0:'non-fire', 1:'wildfire', 2:'false-alarm'}.get(int(lbl), '?')
    print(f'  {lbl} {name:12s}: {cnt:>9,}  ({100*cnt/n_train:.2f}%)')

# Inverse frequency weighting
# w_c = N / (n_classes × n_c)
n_classes = max(unique) + 1  # handles case where class 2 may be absent
raw_weights = {}
for lbl, cnt in zip(unique, counts):
    raw_weights[int(lbl)] = n_train / (n_classes * cnt)

# Ensure all three classes have a weight (even if class 2 has 0 training examples)
for lbl in range(3):
    if lbl not in raw_weights:
        raw_weights[lbl] = raw_weights.get(0, 1.0) * 10  # high weight for missing class

# Normalise so minimum weight = 1.0
min_w = min(raw_weights.values())
class_weights = {lbl: w / min_w for lbl, w in raw_weights.items()}

print(f'\nClass weights (higher = model penalised more for missing that class):')
for lbl, w in sorted(class_weights.items()):
    name = {0:'non-fire', 1:'wildfire', 2:'false-alarm'}.get(lbl, '?')
    bar  = '█' * int(w)
    print(f'  {lbl} {name:12s}: {w:>8.2f}  {bar}')

# Save
weights_path = f'{SCALER_DIR}/class_weights.json'
with open(weights_path, 'w') as f:
    json.dump({'class_weights': class_weights,
               'label_counts': {int(k): int(v) for k, v in zip(unique, counts)},
               'n_train': int(n_train)}, f, indent=2)

print(f'\n✓ Class weights saved → {weights_path}')
print(f'\nUsage in Module 3:')
print(f'  weights_tensor = torch.tensor([{class_weights[0]:.2f}, {class_weights[1]:.2f}, {class_weights.get(2, 20.0):.2f}])')
print(f'  criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(device))')

---
## Section 8 — Update HDF5 with normalised feature vectors

The HDF5 `mlp_atm` and `mlp_srf` arrays were written by Module 1c with raw (unnormalised) values. We update them in-place with the robust-scaled values, and add a new `mlp_derived` dataset for the 6 new derived features.

In [ ]:
# ── Check if mlp_derived already exists in archive ────────────────────────────
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    has_derived = 'mlp_derived' in hf
    n_archive   = hf['labels'].shape[0]

print(f'Archive: {n_archive:,} patches | mlp_derived exists: {has_derived}')

# ── Write normalised features + derived features to HDF5 ──────────────────────
BATCH = 50_000  # process in batches to avoid OOM on Colab

# First create the mlp_derived dataset if missing
if not has_derived:
    with h5py.File(ARCHIVE_PATH, 'a') as hf:
        hf.create_dataset('mlp_derived',
                          shape=(n_archive, len(DERIVED_COLS)),
                          maxshape=(None, len(DERIVED_COLS)),
                          dtype='float32',
                          chunks=(256, len(DERIVED_COLS)))
    print(f'  Created mlp_derived dataset: ({n_archive}, {len(DERIVED_COLS)})')

# Build year → row index mapping from archive
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    archive_years = hf['meta/year'][:]

print('Updating normalised feature vectors in HDF5...')

def safe_feat_array(df, cols, scalers):
    """Extract feature matrix, apply scaling, fill NaN with 0."""
    out = np.zeros((len(df), len(cols)), dtype=np.float32)
    for j, col in enumerate(cols):
        if col not in df.columns:
            continue
        vals = df[col].astype(float).values
        sc   = scalers.get(col, {})
        if sc.get('status') == 'real':
            vals = (vals - sc['median']) / sc['iqr']
        out[:, j] = np.nan_to_num(vals.astype(np.float32), nan=0.0)
    return out


for year in tqdm(ALL_YEARS, desc='Years'):
    if year not in data_v2:
        continue

    year_idxs = np.where(archive_years == year)[0]
    df_yr = data_v2[year].reset_index(drop=True)

    if len(year_idxs) != len(df_yr):
        print(f'  [WARN] {year}: row count mismatch — skipping HDF5 update')
        continue

    atm_arr  = safe_feat_array(df_yr, ATM_FEATURE_COLS, scalers)
    srf_arr  = safe_feat_array(df_yr, SRF_FEATURE_COLS, scalers)
    der_arr  = safe_feat_array(df_yr, DERIVED_COLS,     scalers)

    with h5py.File(ARCHIVE_PATH, 'a') as hf:
        hf['mlp_atm'][year_idxs]     = atm_arr
        hf['mlp_srf'][year_idxs]     = srf_arr
        hf['mlp_derived'][year_idxs] = der_arr

    print(f'  {year}: {len(year_idxs):,} rows updated')

print('\n✓ HDF5 feature vectors updated')

---
## Section 9 — Feature audit visualisation

In [ ]:
# ── Feature quality audit plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor='#0d1117')
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')

# ── Panel 1: Label distribution before and after repair ───────────────────────
ax = axes[0]
before = [55460, 1094262, 0]   # from Module 1c logs
total_b = sum(before)
after_counts = []
for lbl in range(3):
    after_counts.append(int((new_labels == lbl).sum()))

total_a = sum(after_counts)
x = np.arange(3)
w = 0.35
colors_before = ['#4C9BE8', '#E8624C', '#888']
colors_after  = ['#4C9BE8', '#E8624C', '#F5A623']

bars_b = ax.bar(x - w/2, [100*b/total_b for b in before], w,
                color=colors_before, alpha=0.6, label='Before repair')
bars_a = ax.bar(x + w/2, [100*a/total_a for a in after_counts], w,
                color=colors_after, alpha=0.9, label='After repair')

for bar, cnt in zip(bars_a, after_counts):
    if cnt > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{cnt:,}', ha='center', va='bottom', fontsize=7, color='white')

ax.set_xticks(x)
ax.set_xticklabels(['Non-fire\n(label 0)', 'Wildfire\n(label 1)', 'False-alarm\n(label 2)'],
                   color='white', fontsize=9)
ax.set_ylabel('% of dataset', color='white')
ax.set_title('Label Distribution: Before vs After Repair', color='white', fontsize=10, pad=10)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=8)
ax.set_ylim(0, 100)

# ── Panel 2: Feature coverage heatmap ─────────────────────────────────────────
ax = axes[1]
all_feat_cols = ATM_FEATURE_COLS + SRF_FEATURE_COLS + DERIVED_COLS
coverage = []
for col in all_feat_cols:
    sc = scalers.get(col, {})
    coverage.append(1.0 if sc.get('status') == 'real' else 0.0)

# Reshape into grid
n_cols = len(all_feat_cols)
grid_w = 6
grid_h = (n_cols + grid_w - 1) // grid_w
cov_grid = np.zeros(grid_h * grid_w)
cov_grid[:n_cols] = coverage
cov_grid = cov_grid.reshape(grid_h, grid_w)

im = ax.imshow(cov_grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

# Label each cell
for idx, col in enumerate(all_feat_cols):
    r, c = idx // grid_w, idx % grid_w
    short = col[:10]
    color = 'black' if coverage[idx] > 0.5 else 'white'
    ax.text(c, r, short, ha='center', va='center', fontsize=5.5, color=color)

ax.set_title('Feature Coverage\n(green=real data, red=zero-filled)', color='white', fontsize=10, pad=10)
ax.set_xticks([])
ax.set_yticks([])
ax.spines[:].set_color('#444')

# Legend
real_patch  = mpatches.Patch(color='#4CAF50', label=f'Real data ({int(sum(coverage))} features)')
zero_patch  = mpatches.Patch(color='#F44336', label=f'Zero-filled ({n_cols - int(sum(coverage))} features)')
ax.legend(handles=[real_patch, zero_patch], loc='lower right',
          facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=7)

# ── Panel 3: Class weight visualisation ───────────────────────────────────────
ax = axes[2]
w_labels = ['Non-fire\n(label 0)', 'Wildfire\n(label 1)', 'False-alarm\n(label 2)']
w_vals   = [class_weights.get(i, 0) for i in range(3)]
bar_colors = ['#4C9BE8', '#E8624C', '#F5A623']

bars = ax.bar(range(3), w_vals, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
for bar, w in zip(bars, w_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'×{w:.1f}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')

ax.set_xticks(range(3))
ax.set_xticklabels(w_labels, color='white', fontsize=9)
ax.set_ylabel('Loss weight multiplier', color='white')
ax.set_title('Class Weights for Weighted Cross-Entropy Loss\n(higher = harder to miss that class)',
             color='white', fontsize=10, pad=10)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')

ax.text(0.5, 0.97,
        'These weights penalise the model more for missing\nrare classes (non-fire, false-alarm)',
        transform=ax.transAxes, ha='center', va='top', fontsize=8,
        color='#aaa', style='italic')

fig.suptitle('FireSight-IR | Module 2 — Feature Engineering Audit',
             fontsize=12, color='white', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_feature_audit.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Figure saved.')

---
## Section 10 — Final archive verification

In [ ]:
print('=== Final HDF5 Archive Structure ===')
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    def walk(h5, prefix=''):
        for k in h5.keys():
            item = h5[k]
            path = f'{prefix}/{k}'
            if isinstance(item, h5py.Dataset):
                print(f'  {path:<35} {str(item.shape):<30} {item.dtype}')
            else:
                walk(item, path)
    walk(hf)

    labels = hf['labels'][:]
    atm    = hf['mlp_atm'][:, :3]   # first 3 ATM features
    der    = hf['mlp_derived'][:]    # all derived features

print('\n=== Label Distribution (final) ===')
total = len(labels)
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = (labels == lbl).sum()
    print(f'  {lbl} {name:12s}: {n:>9,}  ({100*n/total:.2f}%)')

print('\n=== Normalised ATM feature stats (should be ~N(0,1)) ===')
for j, col in enumerate(ATM_FEATURE_COLS[:3]):
    v = atm[:, j]
    print(f'  {col:<25} mean={v.mean():.3f}  std={v.std():.3f}')

print('\n=== Derived feature stats ===')
for j, col in enumerate(DERIVED_COLS):
    v = der[:, j]
    if v.std() > 0:
        print(f'  {col:<25} mean={v.mean():.3f}  std={v.std():.3f}  '
              f'[{np.percentile(v,1):.2f}, {np.percentile(v,99):.2f}]')
    else:
        print(f'  {col:<25} all zeros (source data unavailable)')

---
## Section 11 — Summary

### What Module 2 produced

| Output | Path | Description |
|---|---|---|
| v2 parquets | `data/processed/viirs_fp_v2/` | 55 → 61 columns, labels repaired |
| Updated archive | `data/processed/patches/firesight_patches.h5` | Labels + normalised features + mlp_derived |
| Scaler params | `data/scalers/feature_scalers.json` | Robust scaler (median+IQR) per feature |
| Class weights | `data/scalers/class_weights.json` | Inverse-frequency weights for Module 3 loss |
| Audit figure | `figures/02_feature_audit.png` | Label repair + feature coverage + class weights |

### How to load in Module 3

```python
import h5py, json, numpy as np
import torch

# Load class weights
with open('data/scalers/class_weights.json') as f:
    cw = json.load(f)
weights = torch.tensor([cw['class_weights'][str(i)] for i in range(3)], dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight=weights.to(device))

# Load features from archive
train_idx = np.load('data/splits/train_index.npy')
with h5py.File('data/processed/patches/firesight_patches.h5', 'r') as hf:
    bt_i4    = hf['cnn/bt_i4'][train_idx]        # (N, 32, 32)  already pseudo-normalised
    mlp_atm  = hf['mlp_atm'][train_idx]           # (N, 16)  robust-scaled
    mlp_srf  = hf['mlp_srf'][train_idx]           # (N, 20)  robust-scaled
    mlp_der  = hf['mlp_derived'][train_idx]        # (N,  6)  robust-scaled
    labels   = hf['labels'][train_idx]            # (N,)  0/1/2
```

### Known limitations going into Module 3

| Feature group | Status | Impact on Module 3 |
|---|---|---|
| ERA5 ATM (16) | ✅ Real, normalised | Full MLP-atm branch working |
| OSM distances (5) | ✅ Real, normalised | Partial MLP-srf working |
| BT pseudo-patches | ✅ Center pixel real | CNN branch working |
| Derived features (6) | ✅ Real (LI, doy, BT SNR) | Additional signal |
| ESA CCI land cover (8) | ❌ All zeros | MLP-srf branch partially blind |
| DNB nighttime lights (2) | ❌ All zeros | False-alarm recovery incomplete |
| Burn scars (2) | ❌ All zeros | LWIR warm-scar not guarded |

The model can still train effectively — the ATM branch alone carries strong signal (fire weather conditions, atmospheric transmission) and the BT pseudo-patches distinguish confirmed fires from background.

### Next: Module 3 — Multi-branch PINN training